# Day 20 — NumPy: **Cosine Similarity from Scratch**

**Phase 1 · Module 3: Prompt Engineering & RAG** · ~30 min

Every vector database — OpenSearch, Pinecone, the Bedrock Knowledge Base you used on Day 18 — ranks results by one number: the **cosine similarity** between the query's embedding and each document's. Today you build it yourself in NumPy, so the retrieval you'll design this afternoon isn't a black box.

Three steps:
1. `cosine_similarity(a, b)` from `np.dot` and `np.linalg.norm`.
2. **Batch** it — one query vector against a whole matrix of embeddings, vectorised.
3. `float32` vs `float64` — the memory/precision trade every embedding store makes.

> **Runnable keyless:** pure NumPy, tiny hand-made vectors standing in for real embeddings. No model, no API key.

## 1. Why cosine, not distance

An embedding is a vector; similar meanings point in **similar directions**. Cosine similarity measures the **angle** between two vectors, ignoring their length — so a short doc and a long doc about the same topic still score high. It's the dot product normalised by both magnitudes:

$$\cos(\theta) = \frac{a \cdot b}{\lVert a \rVert \, \lVert b \rVert}$$

Range: **1** = same direction (identical meaning), **0** = orthogonal (unrelated), **−1** = opposite.

In [1]:
import numpy as np

def cosine_similarity(a, b):
    a, b = np.asarray(a, dtype=np.float32), np.asarray(b, dtype=np.float32)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

same     = cosine_similarity([1, 0, 1], [2, 0, 2])   # same direction, different length
orthogon = cosine_similarity([1, 0, 0], [0, 1, 0])   # unrelated
opposite = cosine_similarity([1, 1], [-1, -1])       # opposite
print(f"same direction : {same:.3f}")
print(f"orthogonal     : {orthogon:.3f}")
print(f"opposite       : {opposite:.3f}")

same direction : 1.000
orthogonal     : 0.000
opposite       : -1.000


☝ `np.dot(a, b)` is the dot product; `np.linalg.norm(v)` is √(Σvᵢ²), the vector's length. `[1,0,1]` and `[2,0,2]` point the same way despite different magnitudes → **1.000**. That length-invariance is exactly why retrieval uses cosine over raw Euclidean distance.

## 2. A tiny RAG retrieval — query vs document embeddings

Give three "policy chunks" 4-dimensional stand-in embeddings and score a query against each. This is the maths inside `FakeKnowledgeBase.retrieve` from Day 18 — and inside real OpenSearch.

In [2]:
# 4-d stand-in embeddings: [mortgage, deposit, overdraft, age] "concept" axes
docs = {
    "POL-002 max LTV 95%":        np.array([0.9, 0.3, 0.0, 0.0]),
    "POL-009 overdraft cap 2000": np.array([0.1, 0.2, 0.9, 0.0]),
    "POL-004 min age 18":         np.array([0.0, 0.1, 0.0, 0.9]),
}
query = np.array([0.85, 0.35, 0.05, 0.0])          # "what is the mortgage LTV limit?"

ranked = sorted(((cosine_similarity(query, v), name) for name, v in docs.items()), reverse=True)
for score, name in ranked:
    print(f"{score:.3f}  {name}")

0.996  POL-002 max LTV 95%
0.234  POL-009 overdraft cap 2000
0.042  POL-004 min age 18


☝ The mortgage query points along the mortgage axis, so **POL-002** scores highest — retrieval is just this, at 1024+ dimensions with millions of docs. Sorting by score gives top-k. Real embeddings come from a model (Titan, `text-embedding-3-small`); the ranking maths is identical.

## 3. Batch it — one query vs a whole matrix

Looping Python over a million documents is slow. Stack all embeddings into a **matrix** and let NumPy compute every score in one vectorised operation — orders of magnitude faster, and the shape a vector DB uses internally.

In [3]:
def cosine_batch(query, matrix):
    query  = np.asarray(query, dtype=np.float32)
    matrix = np.asarray(matrix, dtype=np.float32)
    q_norm = query / np.linalg.norm(query)                      # normalise query once
    m_norm = matrix / np.linalg.norm(matrix, axis=1, keepdims=True)  # normalise each row
    return m_norm @ q_norm                                      # matrix-vector = all dot products

names  = list(docs.keys())
matrix = np.vstack(list(docs.values()))        # shape (3, 4): 3 docs x 4 dims
scores = cosine_batch(query, matrix)

print("scores:", np.round(scores, 3))
top = np.argsort(scores)[::-1]                 # indices, best first
for i in top:
    print(f"{scores[i]:.3f}  {names[i]}")

scores: [0.996 0.234 0.042]
0.996  POL-002 max LTV 95%
0.234  POL-009 overdraft cap 2000
0.042  POL-004 min age 18


☝ `matrix @ q_norm` computes **all** document scores at once (`@` is matrix multiply). `norm(matrix, axis=1, keepdims=True)` normalises each row (document) independently. `np.argsort(...)[::-1]` gives the top-k indices. This vectorised form is what makes vector search fast — no Python loop over documents.

In [4]:
# sanity check: batch matches the per-pair function exactly
loop_scores = [cosine_similarity(query, v) for v in docs.values()]
print("batch == loop:", np.allclose(scores, loop_scores))

batch == loop: True


☝ Same numbers, one operation instead of a loop. On real corpora this is the difference between milliseconds and seconds per query.

## 4. `float32` vs `float64` — the embedding memory trade

NumPy defaults to `float64` (8 bytes/number). Embeddings are stored as `float32` (4 bytes) — **half the memory**, negligible accuracy loss for similarity ranking. At scale this decides whether an index fits in RAM.

In [5]:
dim, n_docs = 1024, 1_000_000                  # a realistic embedding store

for dtype in (np.float64, np.float32):
    bytes_per = np.dtype(dtype).itemsize
    total_gb  = n_docs * dim * bytes_per / 1e9
    print(f"{np.dtype(dtype).name}: {bytes_per} bytes/number -> {total_gb:.1f} GB for {n_docs:,} x {dim}-d")

# precision cost is tiny for ranking
a = np.random.rand(1024)
b = np.random.rand(1024)
c64 = cosine_similarity(a.astype(np.float64), b.astype(np.float64))
c32 = cosine_similarity(a.astype(np.float32), b.astype(np.float32))
print(f"\ncosine  float64={c64:.6f}  float32={c32:.6f}  diff={abs(c64-c32):.2e}")

float64: 8 bytes/number -> 8.2 GB for 1,000,000 x 1024-d
float32: 4 bytes/number -> 4.1 GB for 1,000,000 x 1024-d

cosine  float64=0.745894  float32=0.745894  diff=0.00e+00


☝ `float32` halves the index (4 GB vs 8 GB per million 1024-d vectors) while the similarity score barely moves — a difference far below what affects ranking. This is why every vector store defaults to `float32` (and some push to `int8` for another 4× via quantisation).

## 5. Recap — the maths under vector search

| Tool | Why used |
|---|---|
| `np.dot(a, b)` | dot product — the numerator of cosine |
| `np.linalg.norm(v)` | vector length — normalises out magnitude |
| `cosine_similarity` | direction-based relevance, length-invariant |
| `matrix @ q_norm` | **batch** all document scores in one vectorised op |
| `norm(m, axis=1, keepdims=True)` | per-row (per-doc) normalisation |
| `np.argsort(scores)[::-1]` | top-k indices, best first |
| `float32` vs `float64` | half the memory, negligible ranking loss |

